In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import IPython.display as ipd
import librosa
import librosa.display
import pandas as pd
from scipy.signal import find_peaks
import numpy as np
from typing import List, Dict

In [ ]:
# Load the audio file
x, sr = librosa.load(audio_path, sr=None)

# Play the audio
ipd.Audio(x, rate=sr)
# Plot the waveform
plt.figure(figsize=(14, 5))
librosa.display.waveshow(x, sr=sr, color="steelblue")
plt.ylabel('Amplitude')
plt.title('Waveform')
plt.show()

In [ ]:
onset_times = times[onset_frames]
isi = onset_times * 1000
isi_stim = np.diff(isi)

In [ ]:
def beatGen(TT,isi_range,tap_counters,cycle_length):

    # import necessary libraries
    import numpy as np

    # Parameters

    # BG neuron params
    Vna = 50#sodium reversal potential
    Vca = 50 #calcium reversal potential
    Eh =  -30 #
    Vl = -70 #leak reversal potential
    vm = -40
    km = 6.5
    va = -67
    ka = 1
    vr = -70
    kr = 12
    vh = -60
    kh = 6
    Iint = -33
    C = 1
    gca = 11
    gh = 1
    gna = 0.1
    gl = 1.6
    vrtau = -75
    krtau = 8
    tl = 30
    tr = 5
    trmax = 850


    # S neuron params
    IbiasS = -14
    # IbiasS = 5.25
    gstim = 6
    gcaS = 10
    CS = C
    glS = gl


    deltaT = .2
    deltaP = 2.5
    tx=40
    Sthreshold = -20
    BGthreshold = -20


    dt = .05
    T = int(TT/dt) ## TT is total run time in ms, T is run time in timesteps
    pdist = []

    gammaS = 29
    gammaBG = 0
    gcc = 0
    prevGS =  29

    # intialize dynamic variables to random values
    VS = np.zeros(T) +np.random.uniform(29.81, -77.87) # Stim Voltage
    hS = np.zeros(T)+np.random.uniform(0, 1) # Calcium gate status (0/1) for stimulus
    V = np.zeros(T) +np.random.uniform(28.736, -82.055) # Voltage
    h = np.zeros(T) +np.random.uniform(0, 1) # Calcium current
    r = np.zeros(T) +np.random.uniform(0, 1) # Slow activation variable

    S_pre = np.full(T,0)
    counter = 0


    Istim = np.zeros(T) # Current indicator of stim
    dT = np.zeros(T) # Time step for current
    dP = np.zeros(T) # Time step for Sodium and Phosphorus


    Ibias = np.zeros(T)

    Ibias[0]=5.23779093
    V[0]=-79.97977445
    VS[0]=-76.00277517
    h[0]=0.92376054
    hS[0]=0.89255342
    r[0]=0.1808172

    change=0

    BGspikeTimes = []
    SspikeTimes = []

    # create stim input
    s=0
    for k in range(len(isi_range)):
        stimTimes=np.arange(s,s+isi_range[k],isi_range[k])
        for i in stimTimes:
            Istim[int(i/dt):int((i+25)/dt)] = 1
        s=s+isi_range[k]

    # Initialize x to random value
    x = np.zeros(T) + np.random.uniform(1,2)

    # initialize:
    spikeBG = 0
    spikeS = 0
    sgamma = []
    bggamma = []

    # pred = 1+2*(x1+1)+(x1+1+1)+2+(x1+1)
    pred = cycle_length


    for i,t in enumerate(np.arange(0,TT-dt,dt)):

        ####### Model for BG neuron

        minf = 1/(1+np.exp(-(V[i]-vm)/km))
        ainf = 1/(1+np.exp(-(V[i]-va)/ka))
        rinf = 1/(1+np.exp((V[i]-vr)/kr)) # these are inactivation gates
        hinf = 1/(1+np.exp((V[i]-vh)/kh)) # these are inactivation gates

        tauh = tl/(1+np.exp((V[i]-vh)/kh))+tr/(1+np.exp(-(V[i]-vh)/kh))
        taur = trmax/(np.cosh((V[i]-vrtau)/(2*krtau)))

        dV = 1/1*(Ibias[i]+Iint-gca*minf*h[i]*(V[i]-Vca)-gl*(V[i]-Vl)-gh*r[i]*(V[i]-Eh)-gna*ainf*(V[i]-Vna))
        dh = (hinf-h[i])/tauh
        dr = (rinf-r[i])/taur

        # update state variables
        V[i+1] = V[i] + dV*dt
        h[i+1] = h[i] + dh*dt
        r[i+1] = r[i] + dr*dt


        ######## Model for S neuron

        tauhS = tl/(1+np.exp((VS[i]-vh)/kh))+tr/(1+np.exp(-(VS[i]-vh)/kh))
        minfS = 1/(1+np.exp(-(VS[i]-vm)/km))
        hinfS = 1/(1+np.exp((VS[i]-vh)/kh)) # inactivation gate

        dVS = 1/CS*(IbiasS+gstim*Istim[i]-glS*(VS[i]-Vl)-gcaS*minfS*hS[i]*(VS[i]-Vca))
        dhS = (hinfS - hS[i])/tauhS

        # update state variables
        VS[i+1] = VS[i]+dVS*dt
        hS[i+1] = hS[i]+dhS*dt


        ######## Model for counter neuron

        dx = -x[i]/tx
        x[i+1]  = x[i] + dx*dt

        if x[i+1] < 1:
            gammaS += 1
            gammaBG += 1
            x[i+1] = 2



        ########  Check for BG spikes

        if (V[i+1] >= BGthreshold) and (V[i] < BGthreshold):

            change += deltaT*(gammaBG-prevGS) # period update
            dT[i+1] = deltaT*(gammaBG-prevGS)

            bggamma.append(gammaBG)
            gammaBG = 0
            gcc = 0
            spikeBG = t
            BGspikeTimes.append(t)
            counter+=1
            # print('BGspike: ', counter)

##counter for tapping neuron to learn the Son Clave beats

        if counter<pred:
            if counter in tap_counters:
                # print('S_pre is 1: ', counter)
                S_pre[i]=1
        else:
            if counter%pred==0:
                # print('S_pre is 0: ', counter)
                S_pre[i]=0
                counter=0
            else:
                # print('S_pre is 1 for else: ', counter)
                S_pre[i]=1



        #########   Check for S spikes

        if (VS[i+1] >= Sthreshold) and (VS[i] < Sthreshold):
            spikeS = t
            SspikeTimes.append(t)
            prevGS = gammaS
            sgamma.append(gammaS)
            gammaS = 0

            p = gammaBG/prevGS
            pdist.append(p)



            change +=  deltaP*p*np.abs(1-p)*np.sign(p-0.500001) #phase update
            dP[i+1] = deltaP*p*np.abs(1-p)*np.sign(p-0.500001)

        ########  Update Ibias

        Ibias[i+1] = Ibias[i] + change #update ibias with period and/or phase update
        change = 0
    return BGspikeTimes, SspikeTimes, V, VS,Ibias,h,hS,r,Istim,dT,dP,sgamma,bggamma,S_pre


# In[ ]:

In [ ]:
def TappingNeuron(TT,V_BG,S_pre):

    # import necessary libraries
    import numpy as np

    # Parameters

    C = 20.0      # Capacitance
    I = 80
    gCa = 4.4    # Conductance of Ca channel
    gK = 8.0     # Conductance of K channel
    gL = 2.0     # Conductance of leak channel
    gSyn = 2
    E_exc = 0
    V_Ca = 120.0  # Ca equilibrium potential
    V_K = -84.0   # K equilibrium potential
    V_L = -60.0   # Leak equilibrium potential
    phi = 0.005  # Temperature factor
    V1 = -1.2     # Voltage parameter
    V2 = 18.0     # Voltage parameter
    V3 = 2.0      # Voltage parameter
    V4 = 30.0     # Voltage parameter
    T_rise = 1
    T_decay = 10
    V_theta = 10
    V_theta_syn = -20
    T_left = 1000
    T_right = 1
    counter = 0

    dt = .05
    T = int(TT/dt) ## TT is total run time in ms, T is run time in timesteps

    V = np.zeros(T)  # Voltage
    N = np.zeros(T) # Calcium current
    S = np.zeros(T)  # Slow activation variable
    H = np.zeros(T)

    for j in range(0,len(H)):
        if V_BG[j]<V_theta_syn:
            H[j]=0
        else:
            H[j]=1


    V[0] = -60.0
#     V[0] = -10
    N[0] = 0.05
    S[0] = 1

    TN_spiketimes = []
    binary_signal = []
    TNthreshold = -20

    binary_signal.append(0)

    for i,t in enumerate(np.arange(0,TT-dt,dt)):

        # Model for Tapping neuron

        M_SS = 0.5*(1+np.tanh((V[i]-V1)/V2))
        N_SS = 0.5*(1+np.tanh((V[i]-V3)/V4))

#         T_N = 1/(phi*np.cosh((V[i]-V3)/(2*V4)))


        dV = 1/C*(I-(gL*(V[i]-V_L))-(gCa*M_SS*(V[i]-V_Ca))-(gK*N[i]*(V[i]-V_K))-(gSyn*S[i]*S_pre[i]*(V[i]-E_exc)))

        if V_theta>V[i]:
            T_N=1/(phi*np.cosh((V[i]-V3)/(2*V4)))
        else:
            T_N=T_right

        dN = (N_SS-N[i])/T_N
        dS = (((1-S[i])/T_rise)*(H[i]))-(S[i]/T_decay)

        # update state variables
        V[i+1] = V[i] + dV*dt
        N[i+1] = N[i] + dN*dt
        S[i+1] = S[i] + dS*dt

        if (V[i+1] >= TNthreshold) and (V[i] < TNthreshold):
            TN_spiketimes.append(t)
            binary_signal.append(1)
        else:
            binary_signal.append(0)

    return V,N,S,TN_spiketimes,binary_signal


In [ ]:
# Step 2: Autocorrelation
autocorr = np.correlate(isi_stim - np.mean(isi_stim), isi_stim - np.mean(isi_stim), mode='full')
autocorr = autocorr[autocorr.size // 2:]

# Step 3: Find lag where repetition starts
peaks, _ = find_peaks(autocorr[1:], distance=1)  # Find all peaks after lag 0

if len(peaks) > 0:
    # Get the height of each found peak
    peak_heights = autocorr[1:][peaks]

    # Find the index of the highest peak's height
    highest_peak_index = np.argmax(peak_heights)

    # Get the corresponding lag from the peaks list and add 1 for the slice
    repeat_start = peaks[highest_peak_index] + 1
else:
    # Fallback if no peaks are found
    repeat_start = len(isi_stim)

# Step 4: Extract initial non-repeating segment
initial_segment = isi_stim[:repeat_start]
rounded_segment = np.round(initial_segment).astype(int)

# Step 5: Robust GCD Estimator
def robust_gcd_estimator(y: List[float], margin: int = 50) -> Dict:
    y = np.array(y)
    diffs = [abs(y[i] - y[j]) for i in range(len(y)) for j in range(i+1, len(y))]
    median_diff = int(np.median(diffs))
    g_min = max(1, median_diff - margin)
    g_max = median_diff + margin
    g_range = list(range(g_min, g_max + 1))

    best_gcd = 1
    best_error = float('inf')
    best_multiples = []
    best_reconstruction = []

    for g in g_range:
        m = np.round(y / g)
        y_hat = m * g
        error = np.sum((y - y_hat) ** 2)

        if error < best_error:
            best_error = error
            best_gcd = g
            best_multiples = m.astype(int).tolist()
            best_reconstruction = y_hat.tolist()

    return {
        "input_sequence": y.tolist(),
        "estimated_gcd": best_gcd,
        "estimated_multiples": best_multiples,
        "reconstructed_sequence": best_reconstruction,
        "total_squared_error": best_error
    }

# Step 6: Run the estimator
result = robust_gcd_estimator(rounded_segment.tolist())

# Step 7: Display results
print("Initial Segment (Rounded):", rounded_segment.tolist())
print("Estimated GCD:", result["estimated_gcd"])
print("Estimated Multiples:", result["estimated_multiples"])
print("Reconstructed Sequence:", result["reconstructed_sequence"])
print("Total Squared Error:", result["total_squared_error"])

# Optional: Plot autocorrelation
plt.figure(figsize=(12, 4))
plt.plot(autocorr, label='Autocorrelation')
plt.axvline(repeat_start, color='r', linestyle='--', label='Repeat Start')
plt.title("Autocorrelation of isi_stim")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
bg_isi = round(result_m2["estimated_gcd"])
multiples = np.array(result_m2["estimated_multiples"])

cycle_length = multiples.sum()

# Tap counters (cumulative sum, excluding last overflow)
tap_counters = np.cumsum(np.insert(multiples[:-1], 0, 1))

# New ISI (constant sequence)
new_isi = np.full(200, bg_isi)

# Total time passed
TT_pass = new_isi.sum()

In [ ]:
bgspikes,spikes,V,VS,Ibias,h,hS,r,Istim,dT,dP,sgamma,bggamma,S_pre= beatGen(TT_pass+100,new_isi,tap_counters,cycle_length)
plt.figure(figsize=(16,7))
plt.plot(np.arange(0,TT_pass+100,0.05),VS, label='S neuron')
plt.plot(np.arange(0,TT_pass+100,0.05),V, label='BG neuron')
plt.xlabel('Time (ms)')
plt.ylabel('Voltage')
plt.legend(loc='upper right')
plt.show()

In [ ]:
V_TN,N,S,TNspikes,binary_signal = TappingNeuron(TT_pass+100,V,S_pre)
# V_TN,N,S = TappingNeuron(5000,V)
plt.figure(figsize=(16,7))
plt.plot(np.arange(0,TT_pass+100,0.05),V_TN, label='Tapping neuron')
# plt.plot(np.arange(0,TT_pass+100,0.05),V, label='BG')
plt.xlabel('Time (ms)')
plt.ylabel('Voltage')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# STREAM 1: Robustness & Sensitivity Analysis
import os, json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import find_peaks
from scipy.io import wavfile
import librosa
from typing import List, Dict, Tuple

# ── Consistent plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "serif",
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   11,
    "legend.fontsize":  9,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "figure.dpi":       150,
})

AUDIO_DIR  = "/content/drive/MyDrive/synthetic_audios"
SAVE_DIR   = "/content/drive/MyDrive/Stream1_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Ground-truth registry ────────────────────────────────────────────────────
# Maps filename prefix → (true_multiples, true_gcd_ms_at_base250)
# For variable tempo files the true GCD scales with base_ms (read from filename)
GROUND_TRUTH = {
    "Pattern1_1_1_2":         [1, 1, 2],
    "Pattern2_1_2_2":         [1, 2, 2],
    "Pattern3_1_1_2_1_2":     [1, 1, 2, 1, 2],
    "16beat_Son_clave":        [3, 3, 4, 2, 4],
    "16beat_Rumba_clave":      [3, 4, 3, 2, 4],
    "12beat_Son_clave":        [2, 2, 3, 2, 3],
    "12beat_Rumba_clave":      [2, 3, 2, 2, 3],
}


def load_and_detect_onsets(filepath: str,
                            pad_duration: float = 0.5,
                            min_spacing_s: float = 0.09) -> Tuple[np.ndarray, int]:
    """Load WAV, run onset detection, return (onset_times_ms, sr)."""
    x, sr = librosa.load(filepath, sr=None)
    pad   = np.zeros(int(sr * pad_duration))
    xp    = np.concatenate([pad, x])
    o_env = librosa.onset.onset_strength(y=xp, sr=sr)
    times = librosa.times_like(o_env, sr=sr)
    wait  = int(min_spacing_s * sr / 512)
    frames = librosa.onset.onset_detect(
        onset_envelope=o_env, sr=sr, hop_length=512,
        wait=wait, backtrack=False)
    onset_ms = times[frames] * 1000          # convert to ms
    return onset_ms, sr


def compute_iois(onset_ms: np.ndarray) -> np.ndarray:
    """Differences between successive onset times (ms)."""
    return np.diff(onset_ms)


def autocorr_cycle_length(iois: np.ndarray) -> int:
    """
    Find the first significant autocorrelation peak (lag > 0).
    Returns the estimated cycle length (number of IOIs in one cycle).
    """
    ac = np.correlate(iois - iois.mean(), iois - iois.mean(), mode='full')
    ac = ac[ac.size // 2:]          # keep non-negative lags
    peaks, props = find_peaks(ac[1:], distance=1)
    if len(peaks) == 0:
        return len(iois)            # fallback: whole sequence = one cycle
    best = peaks[np.argmax(ac[1:][peaks])] + 1
    return int(best)


def robust_gcd_estimator(y: List[float], margin: int = 50) -> Dict:
    """Your existing GCD estimator — unchanged."""
    y = np.array(y, dtype=float)
    diffs = [abs(y[i] - y[j]) for i in range(len(y))
                               for j in range(i+1, len(y))]
    if len(diffs) == 0 or np.median(diffs) == 0:
        return {"estimated_gcd": 0, "estimated_multiples": [],
                "reconstructed_sequence": [], "total_squared_error": np.inf,
                "normalized_tse": np.inf}
    median_diff = int(np.median(diffs))
    g_min = max(1, median_diff - margin)
    g_max = median_diff + margin

    best_gcd, best_err, best_m, best_rec = 1, np.inf, [], []
    for g in range(g_min, g_max + 1):
        m    = np.round(y / g)
        yhat = m * g
        err  = np.sum((y - yhat) ** 2)
        if err < best_err:
            best_err, best_gcd = err, g
            best_m   = m.astype(int).tolist()
            best_rec = yhat.tolist()

    n_tse = best_err / len(y)        # normalized TSE (per interval)
    return {"estimated_gcd": best_gcd, "estimated_multiples": best_m,
            "reconstructed_sequence": best_rec,
            "total_squared_error": best_err, "normalized_tse": n_tse}


def parse_filename(fname: str) -> Tuple[str, int]:
    """Extract (pattern_key, base_ms) from filename."""
    base_ms = int(fname.split("_base")[1].replace("ms.wav",""))
    for key in GROUND_TRUTH:
        if fname.startswith(key):
            return key, base_ms
    return "unknown", base_ms


def multiples_correct(estimated: list, true: list) -> bool:
    """True if the estimated multiples match the ground truth."""
    return list(estimated) == list(true)


def gcd_error_ms(estimated_gcd: float, true_gcd: float) -> float:
    return abs(estimated_gcd - true_gcd)


# =============================================================================
# A.  BATCH PIPELINE — run all files, collect results
# =============================================================================
print("=" * 65)
print("A.  BATCH PIPELINE ACROSS ALL SYNTHETIC FILES")
print("=" * 65)

files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")
                and not f.startswith("Pattern4")])   # exclude Pattern4

records = []
for fname in files:
    pkey, base_ms = parse_filename(fname)
    if pkey == "unknown":
        continue
    true_mult = GROUND_TRUTH[pkey]
    true_gcd  = base_ms                              # by construction

    fpath = os.path.join(AUDIO_DIR, fname)
    try:
        onset_ms, sr   = load_and_detect_onsets(fpath)
        iois           = compute_iois(onset_ms)
        cycle_len_est  = autocorr_cycle_length(iois)
        segment        = np.round(iois[:cycle_len_est]).astype(int)
        result         = robust_gcd_estimator(segment.tolist())

        est_gcd   = result["estimated_gcd"]
        est_mult  = result["estimated_multiples"]
        correct   = multiples_correct(est_mult, true_mult)
        gcd_err   = gcd_error_ms(est_gcd, true_gcd)
        ntse      = result["normalized_tse"]

        records.append({
            "file":            fname,
            "pattern":         pkey,
            "base_ms":         base_ms,
            "true_gcd_ms":     true_gcd,
            "true_multiples":  true_mult,
            "est_gcd_ms":      est_gcd,
            "est_multiples":   est_mult,
            "gcd_error_ms":    round(gcd_err, 2),
            "multiples_correct": correct,
            "normalized_tse":  round(ntse, 2),
            "n_onsets":        len(onset_ms),
            "cycle_len_est":   cycle_len_est,
        })
        status = "✓" if correct else "✗"
        print(f"  {status}  {fname:<55}  GCD={est_gcd}ms (true={true_gcd}ms)  "
              f"mult={est_mult}")
    except Exception as e:
        print(f"  ERROR  {fname}: {e}")

df_batch = pd.DataFrame(records)

# ── Summary stats ─────────────────────────────────────────────────────────────
overall_acc = df_batch["multiples_correct"].mean() * 100
mean_gcd_err = df_batch["gcd_error_ms"].mean()
print(f"\n  Overall multiple recovery accuracy : {overall_acc:.1f}%")
print(f"  Mean GCD error (ms)               : {mean_gcd_err:.2f} ms")

# ── Per-pattern accuracy table ────────────────────────────────────────────────
tbl_batch = (df_batch.groupby("pattern")
             .agg(
                 N              = ("file",              "count"),
                 accuracy_pct   = ("multiples_correct", lambda x: 100*x.mean()),
                 mean_gcd_err   = ("gcd_error_ms",      "mean"),
                 mean_ntse      = ("normalized_tse",    "mean"),
             )
             .round(2)
             .reset_index())
print("\n── Per-pattern summary ──────────────────────────────────────────")
print(tbl_batch.to_string(index=False))


# =============================================================================
# B.  GCD RECOVERY ACCURACY vs. NOISE LEVEL
# =============================================================================
print("\n" + "=" * 65)
print("B.  GCD RECOVERY vs. NOISE LEVEL")
print("=" * 65)

# We simulate jittered IOI sequences directly (no audio I/O needed),
# which gives clean experimental control over noise.
# For each pattern and noise level we run 200 Monte-Carlo trials.

PATTERNS_SIM = {
    "Pattern 1\n[1,1,2]":         {"multiples": [1, 1, 2],         "base": 250},
    "Pattern 2\n[1,2,2]":         {"multiples": [1, 2, 2],         "base": 250},
    "Pattern 3\n[1,1,2,1,2]":     {"multiples": [1, 1, 2, 1, 2],  "base": 250},
    "16-beat Son\n[3,3,4,2,4]":   {"multiples": [3, 3, 4, 2, 4],  "base": 250},
    "16-beat Rumba\n[3,4,3,2,4]": {"multiples": [3, 4, 3, 2, 4],  "base": 250},
    "12-beat Son\n[2,2,3,2,3]":   {"multiples": [2, 2, 3, 2, 3],  "base": 250},
    "12-beat Rumba\n[2,3,2,2,3]": {"multiples": [2, 3, 2, 2, 3],  "base": 250},
}

NOISE_LEVELS = [0.03, 0.05, 0.10, 0.20, 0.30, 0.50]  # fraction of IOI std dev
N_TRIALS     = 200
MARGIN       = 50    # GCD search margin in ms

noise_records = []
rng = np.random.default_rng(42)

for pname, pdef in PATTERNS_SIM.items():
    mult = np.array(pdef["multiples"])
    base = pdef["base"]
    true_iois = mult * base

    for noise in NOISE_LEVELS:
        n_correct = 0
        gcd_errors = []
        for _ in range(N_TRIALS):
            # Add Gaussian jitter: noise × base_ms std dev per IOI
            jitter   = rng.normal(0, noise * base, size=len(mult))
            noisy    = np.round(true_iois + jitter).astype(int)
            noisy    = np.clip(noisy, 1, None)   # no negative IOIs
            result   = robust_gcd_estimator(noisy.tolist(), margin=MARGIN)
            est_mult = result["estimated_multiples"]
            correct  = multiples_correct(est_mult, mult.tolist())
            n_correct += int(correct)
            gcd_errors.append(gcd_error_ms(result["estimated_gcd"], base))

        acc = 100 * n_correct / N_TRIALS
        noise_records.append({
            "pattern":    pname,
            "noise_pct":  int(noise * 100),
            "accuracy":   acc,
            "mean_gcd_err_ms": np.mean(gcd_errors),
        })
        print(f"  {pname.split(chr(10))[0]:<20}  noise={int(noise*100):>3}%  "
              f"acc={acc:.1f}%  mean_gcd_err={np.mean(gcd_errors):.1f}ms")

df_noise = pd.DataFrame(noise_records)


# =============================================================================
# C.  GCD SEARCH WINDOW SENSITIVITY  (margin = ±25 / ±50 / ±75 / ±100 ms)
# =============================================================================
print("\n" + "=" * 65)
print("D.  GCD SEARCH WINDOW (MARGIN) SENSITIVITY")
print("=" * 65)

MARGINS   = [25, 50, 75, 100]
NOISE_FIX = 0.10   # fixed 10% noise

margin_records = []
for pname, pdef in PATTERNS_SIM.items():
    mult = np.array(pdef["multiples"])
    base = pdef["base"]
    true_iois = mult * base

    for margin in MARGINS:
        n_correct = 0
        for _ in range(N_TRIALS):
            jitter   = rng.normal(0, NOISE_FIX * base, size=len(mult))
            noisy    = np.round(true_iois + jitter).astype(int)
            noisy    = np.clip(noisy, 1, None)
            result   = robust_gcd_estimator(noisy.tolist(), margin=margin)
            n_correct += int(multiples_correct(result["estimated_multiples"],
                                               mult.tolist()))
        acc = 100 * n_correct / N_TRIALS
        margin_records.append({"pattern": pname, "margin_ms": margin,
                                "accuracy": acc})

df_margin = pd.DataFrame(margin_records)


# =============================================================================
# PLOTS
# =============================================================================

# ── Colour palette ────────────────────────────────────────────────────────────
PAL = ["#2C5F8A", "#E07B39", "#4B9E6F", "#A84C74",
       "#7F5CA8", "#C4A035", "#5AADA3"]

patterns_ordered = list(PATTERNS_SIM.keys())


# ── PLOT B1: Accuracy heatmap (patterns × noise levels) ──────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
pivot = df_noise.pivot(index="pattern", columns="noise_pct", values="accuracy")
# Reorder rows
pivot = pivot.reindex(patterns_ordered)
im = ax.imshow(pivot.values, vmin=0, vmax=100, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c}%" for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([p.replace("\n", " ") for p in pivot.index], fontsize=9)
ax.set_xlabel("Timing jitter (% of base IOI)")
ax.set_title("GCD Multiple Recovery Accuracy (%) vs. Noise Level", pad=10)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                fontsize=9, color="black" if val > 45 else "white",
                fontweight="bold")
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Accuracy (%)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "B1_accuracy_heatmap.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(SAVE_DIR, "B1_accuracy_heatmap.png"), bbox_inches="tight")
plt.show()
print("Saved: B1_accuracy_heatmap")


# ── PLOT B3: Mean GCD error (ms) vs noise ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for i, pname in enumerate(patterns_ordered):
    sub = df_noise[df_noise["pattern"] == pname]
    ax.plot(sub["noise_pct"], sub["mean_gcd_err_ms"],
            marker="s", linewidth=1.8, markersize=5,
            color=PAL[i % len(PAL)],
            label=pname.replace("\n", " "))
ax.axhline(25, color="steelblue", linestyle=":", linewidth=1,
           label="BG tolerance (±25 ms)")
ax.set_xlabel("Timing jitter (% of base IOI)")
ax.set_ylabel("Mean |GCD error| (ms)")
ax.set_title("GCD Estimation Error vs. Noise Level")
ax.legend(fontsize=8, loc="upper left", framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "B3_gcd_error_vs_noise.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(SAVE_DIR, "B3_gcd_error_vs_noise.png"), bbox_inches="tight")
plt.show()
print("Saved: B3_gcd_error_vs_noise")


# ── PLOT D: Margin sensitivity (bar chart) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
x      = np.arange(len(patterns_ordered))
width  = 0.18
colors = ["#C0392B", "#E07B39", "#4B9E6F", "#2C5F8A"]
for j, margin in enumerate(MARGINS):
    vals = [df_margin[(df_margin["pattern"]==p) &
                      (df_margin["margin_ms"]==margin)]["accuracy"].values[0]
            for p in patterns_ordered]
    ax.bar(x + j * width, vals, width, label=f"±{margin} ms", color=colors[j],
           alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([p.replace("\n", " ") for p in patterns_ordered],
                   rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Multiple recovery accuracy (%)")
ax.set_title(f"GCD Search Window Sensitivity\n(fixed {int(NOISE_FIX*100)}% noise, {N_TRIALS} trials)")
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.axhline(95, color="gray", linestyle=":", linewidth=1, alpha=0.7)
ax.legend(title="Search margin", fontsize=9, framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "D_margin_sensitivity.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(SAVE_DIR, "D_margin_sensitivity.png"), bbox_inches="tight")
plt.show()
print("Saved: D_margin_sensitivity")


# =============================================================================
# TABLES  (print + save as CSV)
# =============================================================================

# ── Table 1: Batch pipeline results (compact) ─────────────────────────────────
tbl1 = df_batch[["pattern","base_ms","true_gcd_ms","est_gcd_ms",
                  "gcd_error_ms","est_multiples","multiples_correct",
                  "normalized_tse"]].copy()
tbl1.columns = ["Pattern","Base(ms)","True GCD","Est GCD","GCD Err",
                 "Est Multiples","Correct?","Norm TSE"]
print("\n── Table 1: Batch Pipeline Results ─────────────────────────────────")
print(tbl1.to_string(index=False))
tbl1.to_csv(os.path.join(SAVE_DIR, "Table1_batch_results.csv"), index=False)

# ── Table 2: Noise sensitivity (pivoted) ─────────────────────────────────────
tbl2 = df_noise.pivot_table(index="pattern", columns="noise_pct",
                              values="accuracy").round(1)
tbl2.index = [p.replace("\n"," ") for p in tbl2.index]
tbl2.columns = [f"{c}% noise" for c in tbl2.columns]
print("\n── Table 2: GCD Accuracy (%) vs. Noise Level ────────────────────────")
print(tbl2.to_string())
tbl2.to_csv(os.path.join(SAVE_DIR, "Table2_noise_sensitivity.csv"))



PATTERNS_SIM = {
    "Pattern 1\n[1,1,2]":         {"multiples": [1, 1, 2],         "base": 188},
    "Pattern 2\n[1,2,2]":         {"multiples": [1, 2, 2],         "base": 188},
    "Pattern 3\n[1,1,2,1,2]":     {"multiples": [1, 1, 2, 1, 2],  "base": 188},
    "16-beat Son\n[3,3,4,2,4]":   {"multiples": [3, 3, 4, 2, 4],  "base": 188},
    "16-beat Rumba\n[3,4,3,2,4]": {"multiples": [3, 4, 3, 2, 4],  "base": 188},
    "12-beat Son\n[2,2,3,2,3]":   {"multiples": [2, 2, 3, 2, 3],  "base": 188},
    "12-beat Rumba\n[2,3,2,2,3]": {"multiples": [2, 3, 2, 2, 3],  "base": 188},
}

NOISE_LEVELS = [0.03, 0.05, 0.10, 0.20, 0.30, 0.50]  # fraction of IOI std dev
N_TRIALS     = 200
MARGIN       = 50    # GCD search margin in ms

noise_records = []
rng = np.random.default_rng(42)

# =============================================================================
# D.  GCD SEARCH WINDOW SENSITIVITY  (margin = ±25 / ±50 / ±75 / ±100 ms)
# =============================================================================
print("\n" + "=" * 65)
print("D.  GCD SEARCH WINDOW (MARGIN) SENSITIVITY")
print("=" * 65)

MARGINS   = [25, 50, 75, 100]
NOISE_FIX = 0.10   # fixed 10% noise

margin_records = []
for pname, pdef in PATTERNS_SIM.items():
    mult = np.array(pdef["multiples"])
    base = pdef["base"]
    true_iois = mult * base

    for margin in MARGINS:
        n_correct = 0
        for _ in range(N_TRIALS):
            jitter   = rng.normal(0, NOISE_FIX * base, size=len(mult))
            noisy    = np.round(true_iois + jitter).astype(int)
            noisy    = np.clip(noisy, 1, None)
            result   = robust_gcd_estimator(noisy.tolist(), margin=margin)
            n_correct += int(multiples_correct(result["estimated_multiples"],
                                               mult.tolist()))
        acc = 100 * n_correct / N_TRIALS
        margin_records.append({"pattern": pname, "margin_ms": margin,
                                "accuracy": acc})

df_margin = pd.DataFrame(margin_records)


# ── PLOT D: Margin sensitivity (bar chart) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
x      = np.arange(len(patterns_ordered))
width  = 0.18
colors = ["#C0392B", "#E07B39", "#4B9E6F", "#2C5F8A"]
for j, margin in enumerate(MARGINS):
    vals = [df_margin[(df_margin["pattern"]==p) &
                      (df_margin["margin_ms"]==margin)]["accuracy"].values[0]
            for p in patterns_ordered]
    ax.bar(x + j * width, vals, width, label=f"±{margin} ms", color=colors[j],
           alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([p.replace("\n", " ") for p in patterns_ordered],
                   rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Multiple recovery accuracy (%)")
ax.set_title(f"GCD Search Window Sensitivity\n(fixed {int(NOISE_FIX*100)}% noise, {N_TRIALS} trials)")
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.axhline(95, color="gray", linestyle=":", linewidth=1, alpha=0.7)
ax.legend(title="Search margin", fontsize=9, framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "D_margin_sensitivity_188.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(SAVE_DIR, "D_margin_sensitivity_188.png"), bbox_inches="tight")
plt.show()
print("Saved: D_margin_sensitivity_188")



In [ ]:
# =============================================================================
# STREAM 4: BASELINE COMPARISON

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import librosa
from scipy.signal import find_peaks
from typing import List, Dict, Tuple

# ── Directories ───────────────────────────────────────────────────────────────
AUDIO_DIR = "/content/drive/MyDrive/synthetic_audios"
SAVE_DIR  = "/content/drive/MyDrive/Stream4_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         11,
    "axes.titlesize":    12,
    "axes.labelsize":    11,
    "legend.fontsize":   9,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
})

# ── Ground truth ──────────────────────────────────────────────────────────────
GROUND_TRUTH = {
    "Pattern1_1_1_2":         [1, 1, 2],
    "Pattern2_1_2_2":         [1, 2, 2],
    "Pattern3_1_1_2_1_2":     [1, 1, 2, 1, 2],
    "16beat_Son_clave":        [3, 3, 4, 2, 4],
    "16beat_Rumba_clave":      [3, 4, 3, 2, 4],
    "12beat_Son_clave":        [2, 2, 3, 2, 3],
    "12beat_Rumba_clave":      [2, 3, 2, 2, 3],
}

PAL = {
    "Current method":           "#2C5F8A",
    "Histogram mode":        "#E07B39",
    "Librosa beat tracker":  "#C0392B",
}


# =============================================================================
# SHARED HELPERS  (self-contained copies — safe to re-run)
# =============================================================================

def load_and_detect_onsets(filepath, pad_duration=0.5, min_spacing_s=0.09):
    x, sr = librosa.load(filepath, sr=None)
    pad   = np.zeros(int(sr * pad_duration))
    xp    = np.concatenate([pad, x])
    o_env = librosa.onset.onset_strength(y=xp, sr=sr)
    times = librosa.times_like(o_env, sr=sr)
    wait  = int(min_spacing_s * sr / 512)
    frames = librosa.onset.onset_detect(
        onset_envelope=o_env, sr=sr, hop_length=512,
        wait=wait, backtrack=False)
    return times[frames] * 1000, x, sr   # onset_ms, raw signal, sample rate

def compute_iois(onset_ms):
    return np.diff(onset_ms)

def autocorr_cycle_length(iois):
    ac = np.correlate(iois - iois.mean(), iois - iois.mean(), mode='full')
    ac = ac[ac.size // 2:]
    peaks, _ = find_peaks(ac[1:], distance=1)
    if len(peaks) == 0:
        return len(iois)
    return int(peaks[np.argmax(ac[1:][peaks])] + 1)

def robust_gcd_estimator(y: List[float], margin: int = 50) -> Dict:
    y = np.array(y, dtype=float)
    diffs = [abs(y[i] - y[j]) for i in range(len(y))
                               for j in range(i+1, len(y))]
    if len(diffs) == 0 or np.median(diffs) == 0:
        return {"estimated_gcd": 0, "estimated_multiples": [],
                "reconstructed_sequence": [], "total_squared_error": np.inf}
    median_diff = int(np.median(diffs))
    g_min, g_max = max(1, median_diff - margin), median_diff + margin
    best_gcd, best_err, best_m, best_rec = 1, np.inf, [], []
    for g in range(g_min, g_max + 1):
        m    = np.round(y / g)
        yhat = m * g
        err  = float(np.sum((y - yhat) ** 2))
        if err < best_err:
            best_err, best_gcd = err, g
            best_m   = m.astype(int).tolist()
            best_rec = yhat.tolist()
    return {"estimated_gcd": best_gcd, "estimated_multiples": best_m,
            "reconstructed_sequence": best_rec,
            "total_squared_error": best_err}

def build_tap_counters(multiples, cycle_length):
    tap_counters = [1]
    i = 1
    for j in multiples:
        if i + j < cycle_length:
            tap_counters.append(i + j)
            i += j
    return tap_counters

def find_lockin_time(Ibias, BGspikeTimes, dt=0.05,
                     stability_threshold=0.02, min_stable_cycles=3):
    if len(BGspikeTimes) < min_stable_cycles + 1:
        return BGspikeTimes[0] if BGspikeTimes else 0.0
    ibias_at_spikes = [Ibias[int(t / dt)] for t in BGspikeTimes
                       if int(t / dt) < len(Ibias)]
    stable_count = 0
    for k in range(1, len(ibias_at_spikes)):
        delta = abs(ibias_at_spikes[k] - ibias_at_spikes[k-1])
        if delta < stability_threshold:
            stable_count += 1
            if stable_count >= min_stable_cycles:
                return BGspikeTimes[k - min_stable_cycles]
        else:
            stable_count = 0
    return BGspikeTimes[int(len(BGspikeTimes) * 0.6)]

def compute_alignment_metrics(TN_spikes, audio_onset_ms, gcd_ms):
    if len(TN_spikes) == 0 or len(audio_onset_ms) == 0:
        return {"mean_align_err_ms": np.nan, "sd_align_err_ms": np.nan,
                "mean_phase_err": np.nan, "n_spikes": 0}
    errors = [abs(s - audio_onset_ms[np.argmin(np.abs(audio_onset_ms - s))])
              for s in TN_spikes]
    errors = np.array(errors)
    return {"mean_align_err_ms": float(np.mean(errors)),
            "sd_align_err_ms":   float(np.std(errors)),
            "mean_phase_err":    float(np.mean(errors / gcd_ms)),
            "n_spikes":          len(errors)}

def tse_and_multiples(iois, gcd):
    """Compute TSE and multiples given a GCD estimate."""
    iois  = np.array(iois, dtype=float)
    m     = np.round(iois / gcd).astype(int)
    yhat  = m * gcd
    tse   = float(np.sum((iois - yhat) ** 2))
    ntse  = tse / len(iois)
    return m.tolist(), round(tse, 2), round(ntse, 2)


# =============================================================================
# PART 4a — THREE METHOD COMPARISON
# =============================================================================
# ── Method 2: IOI Histogram Mode ─────────────────────────────────────────────
# The simplest possible baseline. Build a histogram of all IOI values,
# find the most common bin (the mode), use that as the GCD estimate.
# Works if the most common IOI happens to be the base unit, but fails when
# the short IOI is rare (e.g. Pattern 2 [1,2,2] — the "1" occurs only once
# per cycle so the mode will be the "2" interval instead).

def histogram_mode_gcd(iois: np.ndarray, bin_width_ms: float = 20.0) -> Dict:
    """
    Estimate GCD as the mode of the IOI histogram.
    bin_width_ms: width of each histogram bin in ms.
    """
    iois = np.array(iois, dtype=float)
    bins = np.arange(0, iois.max() + bin_width_ms, bin_width_ms)
    counts, edges = np.histogram(iois, bins=bins)
    mode_bin   = np.argmax(counts)
    gcd_est    = float((edges[mode_bin] + edges[mode_bin + 1]) / 2)
    gcd_est    = max(gcd_est, 1.0)
    multiples, tse, ntse = tse_and_multiples(iois, gcd_est)
    return {"estimated_gcd": round(gcd_est, 1),
            "estimated_multiples": multiples,
            "total_squared_error": tse,
            "normalized_tse": ntse}


# ── Method 3: Librosa Beat Tracker ───────────────────────────────────────────
# Librosa's beat_track() estimates a global tempo (BPM) and returns beat
# frames. We convert the estimated tempo to a beat interval in ms, then
# use that as the GCD candidate. This is the standard music-information-
# retrieval approach — it works well for isochronous rhythms but was not
# designed for non-isochronous patterns like claves.

def librosa_beat_gcd(x: np.ndarray, sr: int, iois: np.ndarray) -> Dict:
    """
    Estimate GCD using Librosa's beat tracker tempo estimate.
    """
    tempo, _ = librosa.beat.beat_track(y=x, sr=sr)
    # tempo is in BPM — convert to ms per beat
    if hasattr(tempo, '__len__'):   # newer librosa returns array
        tempo = float(tempo[0])
    beat_ms = 60000.0 / float(tempo)
    # beat_ms is the beat interval; our GCD is likely a subdivision of this
    # Try beat_ms and its halvings (½, ¼) and pick the one with lowest TSE
    candidates = [beat_ms, beat_ms / 2, beat_ms / 4, beat_ms / 3]
    best_gcd, best_tse, best_m = beat_ms, np.inf, []
    for c in candidates:
        if c < 50:   # GCD below 50ms is not musically meaningful here
            continue
        m, tse, _ = tse_and_multiples(iois, c)
        if tse < best_tse:
            best_tse, best_gcd, best_m = tse, c, m
    _, tse_final, ntse_final = tse_and_multiples(iois, best_gcd)
    return {"estimated_gcd": round(best_gcd, 1),
            "estimated_multiples": best_m,
            "total_squared_error": tse_final,
            "normalized_tse": ntse_final,
            "raw_tempo_bpm": round(float(tempo), 1)}


# ── Run all three methods on all files ───────────────────────────────────────
print("=" * 70)
print("PART 4a — THREE-METHOD COMPARISON")
print("=" * 70)

files_all = sorted([f for f in os.listdir(AUDIO_DIR)
                    if f.endswith(".wav") and not f.startswith("Pattern4")])

records_4a = []

for fname in files_all:
    pkey = next((k for k in GROUND_TRUTH if fname.startswith(k)), None)
    if pkey is None:
        continue
    base_ms   = int(fname.split("_base")[1].replace("ms.wav", ""))
    true_mult = GROUND_TRUTH[pkey]
    true_gcd  = base_ms

    fpath = os.path.join(AUDIO_DIR, fname)
    onset_ms, x_raw, sr = load_and_detect_onsets(fpath)
    iois = compute_iois(onset_ms)

    # ── Method 1: Proposed method ─────────────────────────────────────────────
    cycle_len = autocorr_cycle_length(iois)
    segment   = np.round(iois[:cycle_len]).astype(int)
    r1        = robust_gcd_estimator(segment.tolist())
    m1_correct = r1["estimated_multiples"] == true_mult
    m1_gcderr  = abs(r1["estimated_gcd"] - true_gcd)
    m1_ntse    = round(r1["total_squared_error"] / len(segment), 2)

    # ── Method 2: Histogram mode ──────────────────────────────────────────
    r2        = histogram_mode_gcd(iois)
    # use the same cycle-length segment for a fair comparison
    m2_iois   = np.round(iois[:cycle_len]).astype(int)
    m2_mult, m2_tse, m2_ntse = tse_and_multiples(
        m2_iois, r2["estimated_gcd"])
    m2_correct = m2_mult == true_mult
    m2_gcderr  = abs(r2["estimated_gcd"] - true_gcd)

    # ── Method 3: Librosa beat tracker ───────────────────────────────────
    r3        = librosa_beat_gcd(x_raw, sr, np.round(iois[:cycle_len]).astype(float))
    m3_correct = r3["estimated_multiples"] == true_mult
    m3_gcderr  = abs(r3["estimated_gcd"] - true_gcd)
    m3_ntse    = r3["normalized_tse"]

    print(f"\n  {fname}")
    print(f"    True:      mult={true_mult}  GCD={true_gcd}ms")
    print(f"    Method 1:  mult={r1['estimated_multiples']}  "
          f"GCD={r1['estimated_gcd']}ms  err={m1_gcderr:.1f}ms  "
          f"nTSE={m1_ntse:.1f}  {'✓' if m1_correct else '✗'}")
    print(f"    Method 2:  mult={m2_mult}  "
          f"GCD={r2['estimated_gcd']}ms  err={m2_gcderr:.1f}ms  "
          f"nTSE={m2_ntse:.1f}  {'✓' if m2_correct else '✗'}")
    print(f"    Method 3:  mult={r3['estimated_multiples']}  "
          f"GCD={r3['estimated_gcd']}ms  err={m3_gcderr:.1f}ms  "
          f"nTSE={m3_ntse:.1f}  {'✓' if m3_correct else '✗'}")

    for method, correct, gcderr, ntse in [
        ("Current method",          m1_correct, m1_gcderr, m1_ntse),
        ("Histogram mode",       m2_correct, m2_gcderr, m2_ntse),
        ("Librosa beat tracker", m3_correct, m3_gcderr, m3_ntse),
    ]:
        records_4a.append({
            "file":       fname,
            "pattern":    pkey,
            "base_ms":    base_ms,
            "method":     method,
            "correct":    correct,
            "gcd_err_ms": round(gcderr, 2),
            "norm_tse":   round(ntse, 2),
        })

df_4a = pd.DataFrame(records_4a)

# ── Accuracy summary table ────────────────────────────────────────────────────
df_acc = (df_4a.groupby(["pattern", "method"])
               .agg(accuracy  = ("correct",    lambda x: 100 * x.mean()),
                    mean_gcderr = ("gcd_err_ms", "mean"),
                    mean_ntse   = ("norm_tse",   "mean"))
               .round(2)
               .reset_index())

print("\n── Accuracy Summary (%) ────────────────────────────────────────────")
print(df_acc.pivot(index="pattern", columns="method",
                   values="accuracy").round(1).to_string())

df_4a.to_csv(os.path.join(SAVE_DIR, "4a_method_comparison_full.csv"),
             index=False)
df_acc.to_csv(os.path.join(SAVE_DIR, "4a_method_comparison_summary.csv"),
              index=False)


# ── PLOT 4a-1: Accuracy grouped bar chart ────────────────────────────────────
patterns_list = df_4a["pattern"].unique().tolist()
methods       = ["Current method", "Histogram mode", "Librosa beat tracker"]
colors_bar    = [PAL[m] for m in methods]

x     = np.arange(len(patterns_list))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
for j, method in enumerate(methods):
    vals = []
    for pname in patterns_list:
        sub = df_4a[(df_4a["pattern"] == pname) & (df_4a["method"] == method)]
        vals.append(100 * sub["correct"].mean())
    ax.bar(x + j * width, vals, width, label=method,
           color=colors_bar[j], alpha=0.87)

ax.set_xticks(x + width)
ax.set_xticklabels([p.replace("_", " ") for p in patterns_list],
                   rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Multiple recovery accuracy (%)")
ax.set_title("GCD Method Comparison: Multiple Recovery Accuracy\n"
             "Across All Patterns and Tempos")
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.axhline(95, color="gray", linestyle=":", linewidth=1, alpha=0.7)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "4a_accuracy_comparison.pdf"),
            bbox_inches="tight")
plt.savefig(os.path.join(SAVE_DIR, "4a_accuracy_comparison.png"),
            bbox_inches="tight")
plt.show()
print("Saved: 4a_accuracy_comparison")


In [ ]:
# =============================================================================
# STREAM 5: All 7 Patterns — Full Three-Safeguard
# =============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.signal import find_peaks

SAVE_DIR = "/content/drive/MyDrive/Stream5_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    10,
    "axes.labelsize":    9,
    "legend.fontsize":   8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        150,
})

# ── Patterns ──────────────────────────────────────────────────────────────────
PATTERNS = [
    ("Pattern 1\n[1,1,2]",          [1,1,2]),
    ("Pattern 2\n[1,2,2]",          [1,2,2]),
    ("Pattern 3\n[1,1,2,1,2]",      [1,1,2,1,2]),
    ("16-beat Son\n[3,3,4,2,4]",    [3,3,4,2,4]),
    ("16-beat Rumba\n[3,4,3,2,4]",  [3,4,3,2,4]),
    ("12-beat Son\n[2,2,3,2,3]",    [2,2,3,2,3]),
    ("12-beat Rumba\n[2,3,2,2,3]",  [2,3,2,2,3]),
]

BASE_MS    = 250
NOISE      = 0.20     # stress-test level — makes all ambiguities visible
N_CYCLES   = 3
THRESH     = 0.3      # S1 prominence threshold
SIM_THRESH = 0.85     # S2 segment similarity threshold

# Fixed seed → exactly reproducible, matches pre-computed annotations
rng = np.random.default_rng(99)

# ── Colours ───────────────────────────────────────────────────────────────────
C_LINE  = "#2C5F8A"   # autocorrelation curve
C_THOLD = "#E07B39"   # prominence threshold
C_TRUE  = "#27AE60"   # correct lag
C_SUB   = "#C0392B"   # subharmonic risk
C_HARM  = "#7F5CA8"   # harmonic risk
C_WEAK  = "#AAAAAA"   # below-threshold peak (harmless)

# ── Helpers ───────────────────────────────────────────────────────────────────
def seg_sim(iois, lag):
    if lag <= 0 or 2 * lag > len(iois):
        return 0.0
    s1    = iois[:lag].astype(float)
    s2    = iois[lag:2*lag].astype(float)
    scale = np.max(np.abs(iois)) + 1e-9
    return float(max(1.0 - np.mean(np.abs(s1 - s2)) / scale, 0.0))


def run_safeguards(iois, ac_norm):
    """Apply S1 → S2 → S3 and return full decision trace."""
    peaks, _ = find_peaks(ac_norm[1:], distance=1)
    peaks     = (peaks + 1).tolist()

    # S1: prominence filter
    above_s1 = [p for p in peaks if ac_norm[p] >= THRESH]

    # S2: segment similarity filter
    sims      = {p: seg_sim(iois, p) for p in above_s1}
    above_s12 = [p for p in above_s1 if sims[p] >= SIM_THRESH]
    s2_relaxed = False
    if not above_s12:
        above_s12  = above_s1   # relax S2 — keep all S1 survivors
        s2_relaxed = True

    # S3: pick shortest valid lag
    chosen = min(above_s12)

    # Determine which safeguard was decisive
    if len(above_s1) == 1:
        decisive = "S1"
    elif above_s12 != above_s1 and not s2_relaxed:
        decisive = "S2"
    else:
        decisive = "S3"

    return {
        "peaks":      peaks,
        "above_s1":   above_s1,
        "above_s12":  above_s12,
        "sims":       sims,
        "chosen":     chosen,
        "decisive":   decisive,
        "s2_relaxed": s2_relaxed,
    }


# ── Pre-compute all data (single rng pass → same sequences for all plots) ─────
all_data = []
for name, mult in PATTERNS:
    mult_arr = np.array(mult)
    true_len = len(mult)
    iois     = np.tile(mult_arr, N_CYCLES).astype(float) * BASE_MS
    iois    += rng.normal(0, NOISE * BASE_MS, size=len(iois))

    ac      = np.correlate(iois - iois.mean(), iois - iois.mean(), mode='full')
    ac      = ac[ac.size // 2:]
    ac_norm = ac / (ac[0] + 1e-9)

    dec = run_safeguards(iois, ac_norm)

    all_data.append({
        "name":     name,
        "mult":     mult_arr,
        "true_len": true_len,
        "iois":     iois,
        "ac_norm":  ac_norm,
        **dec,
    })

# =============================================================================
# CORE PLOTTING FUNCTION
# =============================================================================

def draw_panel(ax, d, show_xlabel=True, show_ylabel=True):
    """Draw one fully annotated autocorrelation panel."""
    true_len = d["true_len"]
    ac_norm  = d["ac_norm"]
    iois     = d["iois"]
    peaks    = d["peaks"]
    above_s1 = d["above_s1"]
    above_s12= d["above_s12"]
    sims     = d["sims"]
    chosen   = d["chosen"]
    decisive = d["decisive"]
    x_limit  = true_len * 2 + 2

    lags = np.arange(len(ac_norm))

    # ── Autocorrelation curve ────────────────────────────────────────────────
    ax.plot(lags[1:x_limit+1], ac_norm[1:x_limit+1],
            color=C_LINE, linewidth=1.8, zorder=3)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.3, zorder=1)

    # ── S1: prominence threshold band ────────────────────────────────────────
    ax.axhline(THRESH, color=C_THOLD, linestyle="--",
               linewidth=1.4, zorder=4, label=f"S1 threshold ({THRESH})")
    ax.fill_between(lags[1:x_limit+1], -1, THRESH,
                    alpha=0.06, color=C_THOLD, zorder=0)
    ax.text(x_limit + 0.1, THRESH + 0.03,
            " ", ha="left", va="bottom",
            fontsize=6.5, color=C_THOLD, style="italic",
            transform=ax.get_xaxis_transform() if False else ax.transData)

    # ── Draw all peaks ────────────────────────────────────────────────────────
    in_range = [p for p in peaks if 1 <= p <= x_limit]

    for p in in_range:
        h    = ac_norm[p]
        sim  = sims.get(p, seg_sim(iois, p))
        is_true  = (p == true_len)
        is_harm  = (p == true_len * 2)
        is_sub   = (p < true_len)
        in_s1    = (p in above_s1)
        in_s12   = (p in above_s12)

        # ── Colour logic ─────────────────────────────────────────────────────
        if is_true:
            col = C_TRUE
            ms  = 130
        elif in_s1 and is_sub:
            col = C_SUB     # above S1 threshold AND subharmonic → real risk
            ms  = 100
        elif in_s1 and is_harm:
            col = C_HARM    # above S1 threshold AND harmonic → real risk
            ms  = 100
        elif in_s1:
            col = C_HARM    # above S1 but not sub/harm — still a competitor
            ms  = 80
        else:
            col = C_WEAK    # below threshold — already dead
            ms  = 50

        ax.scatter([p], [h], color=col, s=ms, zorder=5 + is_true,
                   alpha=1.0 if in_s1 or is_true else 0.5)

        # ── Green vertical line at true lag ───────────────────────────────────
        if is_true:
            ax.axvline(p, color=C_TRUE, linewidth=1.6,
                       linestyle="-", alpha=0.75, zorder=2)


    # ── Axes formatting ───────────────────────────────────────────────────────
    title_clean = d["name"].replace("\n", "  ")
    ax.set_title(title_clean, fontsize=10, fontweight="bold")
    ax.set_xlim(0.3, x_limit + 0.5)
    ax.set_ylim(-0.92, 0.98)
    if show_xlabel:
        ax.set_xlabel("Lag (number of IOIs)", fontsize=9)
    if show_ylabel:
        ax.set_ylabel("Normalised autocorrelation", fontsize=9)
    ax.grid(axis="y", alpha=0.2)


# =============================================================================
# INDIVIDUAL PLOTS
# =============================================================================

LEGEND_ELEMENTS = [
    Line2D([0],[0], color=C_LINE, lw=2),
    Line2D([0],[0], color=C_THOLD, lw=1.5, ls="--"),
    Patch(facecolor=C_THOLD, alpha=0.12),
    Line2D([0],[0], color=C_TRUE, lw=2),
    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=C_SUB, markersize=9),
    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=C_HARM, markersize=9),
    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=C_WEAK, markersize=8, alpha=0.5),
]

for d in all_data:
    fig, ax = plt.subplots(figsize=(10, 5))
    draw_panel(ax, d, show_xlabel=True, show_ylabel=True)
    # ax.legend(handles=LEGEND_ELEMENTS, fontsize=8, loc="lower right",
              # framealpha=0.93, ncol=1)
    fig.suptitle(
        f"Three-Safeguard Autocorrelation Analysis\n"
        f"{d['name'].replace(chr(10),' ')}   |   "
        f"Base GCD = {BASE_MS} ms,  {int(NOISE*100)}% noise,  "
        f"{N_CYCLES} cycles",
        fontsize=11, y=1.01
    )
    plt.tight_layout()
    safe = (d["name"].replace("\n","_").replace(" ","_")
                     .replace("[","").replace("]","").replace(",",""))
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(SAVE_DIR, f"Stream5_new_{safe}.{ext}"),
                    bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"  Saved: Stream5_new_{safe}  "
          f"[chosen=lag {d['chosen']}, decisive={d['decisive']}]")


# =============================================================================
# COMBINED 7-PANEL FIGURE
# =============================================================================

fig = plt.figure(figsize=(20, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.40)
axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(4)]

for idx, d in enumerate(all_data):
    show_x = (idx >= 3)    # x-label only on bottom row
    show_y = (idx % 4 == 0) # y-label only on left column
    draw_panel(axes[idx], d, show_xlabel=show_x, show_ylabel=show_y)

# ── Legend panel (8th cell) ───────────────────────────────────────────────────
ax_leg = axes[7]
ax_leg.axis("off")

fig.suptitle(
    f"Three-Safeguard Autocorrelation Analysis\n"
    f"Base GCD = {BASE_MS} ms  |  {int(NOISE*100)}% timing noise  |  ",
    fontsize=13, y=1.01
)
for ext in ("pdf", "png"):
    plt.savefig(os.path.join(SAVE_DIR, f"Stream5_new_all7_combined.{ext}"),
                bbox_inches="tight")
plt.show()
plt.close()
print("\n  Saved: Stream5_new_all7_combined")
print(f"\n✓  All files in: {SAVE_DIR}")